<a href="https://colab.research.google.com/github/theiman112860/Data-Science-Repository/blob/main/Converting_Swedish_document_to_English.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This code converts a pdf to word doc, installs and loads ollama and a translation llm, and converts a book in swedish to english

In [ ]:
# ============================================================
# COMPLETE COLAB CELL (copy/paste)
# PDF -> DOCX -> Translate (Ollama TranslateGemma local) -> PDF
# + total paragraph counting
# + rolling ETA
# + checkpointing + RESUME after Colab disconnect
# + IMPORTANT CHANGE: state + output DOCX are stored DIRECTLY on Google Drive
#   so you never lose progress, and resumes work automatically.
# ============================================================

# ---------- A) System + Python deps ----------
!apt-get -qq update
!apt-get -qq install -y zstd libreoffice > /dev/null
!pip -q install "requests==2.32.4" python-docx pdf2docx

import os, re, json, time, shutil, requests, statistics
from pathlib import Path
from collections import deque
from docx import Document
from pdf2docx import Converter

# ---------- B) Mount Google Drive (PERSISTENT storage) ----------
from google.colab import drive
drive.mount("/content/drive")

CKPT_DIR = "/content/drive/MyDrive/superminnet_ckpt"
Path(CKPT_DIR).mkdir(parents=True, exist_ok=True)
print("✅ Persistent checkpoint dir:", CKPT_DIR)

# ---------- C) Paths ----------
# Source PDF (in Colab VM) — change if needed
PDF_PATH  = "/content/sample_data/Superminnets hemligheter3.pdf"

# Intermediate DOCX (can stay in VM; we'll regenerate if needed)
DOCX_SV   = "/content/superminnet_sv.docx"

# OUTPUTS + STATE stored directly on Drive (so they survive restarts)
DOCX_EN   = f"{CKPT_DIR}/superminnet_en.docx"
STATE_JSON = f"{CKPT_DIR}/translate_state.json"
TITLE_CACHE_JSON = f"{CKPT_DIR}/title_cache.json"

# Final PDF output (we create in /content then copy to Drive)
PDF_EN_LOCAL = "/content/superminnet_en.pdf"
PDF_EN_DRIVE = f"{CKPT_DIR}/superminnet_en.pdf"

OLLAMA_LOG = "/content/ollama.log"

# ---------- D) Install Ollama + start server ----------
!curl -fsSL https://ollama.com/install.sh -o install.sh
!bash install.sh

os.environ["PATH"] = "/usr/local/bin:" + os.environ.get("PATH", "")
print("ollama binary:", shutil.which("ollama"))
!ollama --version

# Start server (safe if already running)
!nohup ollama serve > /content/ollama.log 2>&1 &
time.sleep(2)

# Health check
try:
    r = requests.get("http://127.0.0.1:11434", timeout=3)
    print("✅ Ollama status:", r.status_code)
except Exception as e:
    print("❌ Ollama not responding:", e)
    !tail -n 80 /content/ollama.log

# Pull translation model
!ollama pull translategemma:4b
MODEL = "translategemma:4b"
OLLAMA_URL = "http://127.0.0.1:11434/api/generate"

# ---------- E) Convert PDF -> DOCX (only if needed) ----------
# If DOCX_SV doesn't exist (new VM), regenerate it.
if not Path(DOCX_SV).exists():
    print("Converting PDF -> DOCX:", PDF_PATH)
    cv = Converter(PDF_PATH)
    cv.convert(DOCX_SV, start=0, end=None)  # whole PDF
    cv.close()
    print("✅ Wrote:", DOCX_SV)
else:
    print("✅ Found existing:", DOCX_SV)

# ---------- F) Glossary (EDIT / EXPAND) ----------
GLOSSARY = {
    "Superminnets hemligheter": "The Secrets of Supermemory",
    "Superminnet": "Supermemory",
    "GMS-metoden": "the GMS method",
    "GMS": "GMS",
}

# ---------- G) Heuristics ----------
CHAPTER_RE = re.compile(r"^\s*(kapitel|chapter)\s+([0-9ivxlcdm]+)\b|^\s*[0-9]+\.\s+", re.IGNORECASE)

# ---------- H) JSON helpers ----------
def load_json(path, default):
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return default

def save_json(path, obj):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

# Load title cache from Drive (or start empty)
title_cache = load_json(TITLE_CACHE_JSON, {})

# ---------- I) DOCX traversal ----------
def iter_all_paragraphs(doc: Document):
    # Body paragraphs
    for p in doc.paragraphs:
        yield p
    # Tables in body
    for table in doc.tables:
        for row in table.rows:
            for cell in row.cells:
                for p in cell.paragraphs:
                    yield p
    # Headers/footers (and their tables)
    for section in doc.sections:
        parts = [
            section.header, section.footer,
            section.first_page_header, section.first_page_footer,
            section.even_page_header, section.even_page_footer,
        ]
        for part in parts:
            if not part:
                continue
            for p in part.paragraphs:
                yield p
            for table in part.tables:
                for row in table.rows:
                    for cell in row.cells:
                        for p in cell.paragraphs:
                            yield p

def replace_paragraph_text_preserve_style(p, new_text: str):
    # Preserve paragraph formatting; keep style of first run; clear others
    if p.runs:
        p.runs[0].text = new_text
        for r in p.runs[1:]:
            r.text = ""
    else:
        p.add_run(new_text)

def is_heading_like(p):
    style = (p.style.name if p.style else "") or ""
    txt = (p.text or "").strip()
    if not txt:
        return False
    if "Heading" in style:
        return True
    if CHAPTER_RE.search(txt):
        return True
    if len(txt) <= 80 and txt.isupper() and any(c.isalpha() for c in txt):
        return True
    return False

# ---------- J) Chunking (prevents timeouts on mega-paragraphs) ----------
def normalize_space(s: str) -> str:
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s

def split_for_translation(text: str, max_chars=1000):
    """
    Split text into chunks that finish reliably on Colab CPU.
    Prefer: blank lines -> sentences -> hard cut.
    """
    t = normalize_space(text).strip()
    if len(t) <= max_chars:
        return [t]

    blocks = re.split(r"\n\s*\n", t)
    chunks, buf = [], ""
    for b in blocks:
        b = b.strip()
        if not b:
            continue
        cand = (buf + "\n\n" + b).strip() if buf else b
        if len(cand) <= max_chars:
            buf = cand
        else:
            if buf:
                chunks.append(buf)
                buf = ""
            if len(b) <= max_chars:
                buf = b
            else:
                sentences = re.split(r"(?<=[\.\!\?])\s+", b)
                sbuf = ""
                for s in sentences:
                    s = s.strip()
                    if not s:
                        continue
                    cand2 = (sbuf + " " + s).strip() if sbuf else s
                    if len(cand2) <= max_chars:
                        sbuf = cand2
                    else:
                        if sbuf:
                            chunks.append(sbuf)
                        while len(s) > max_chars:
                            chunks.append(s[:max_chars])
                            s = s[max_chars:]
                        sbuf = s
                if sbuf:
                    chunks.append(sbuf)
    if buf:
        chunks.append(buf)
    return chunks

# ---------- K) Glossary placeholders (exact enforcement) ----------
def apply_glossary_placeholders(src: str, glossary: dict):
    items = sorted(glossary.items(), key=lambda kv: len(kv[0]), reverse=True)
    ph_map, protected = {}, src
    for i, (sv, en) in enumerate(items, start=1):
        token = f"@@GLOSS_{i}@@"
        if sv in protected:
            protected = protected.replace(sv, token)
            ph_map[token] = en
    return protected, ph_map

def restore_glossary_placeholders(translated: str, ph_map: dict):
    out = translated
    for token, en in ph_map.items():
        out = out.replace(token, en)
    return out

# ---------- L) Ollama translate call + guards ----------
def ollama_translate_once(text: str, timeout_s=900, temperature=0.1, strong=False) -> str:
    if strong:
        prompt = (
            "Translate Swedish to English. You MUST translate all words (including titles).\n"
            "Output ONLY the translation.\n"
            "Keep line breaks, numbers, and punctuation.\n\n"
            f"TEXT:\n{text}"
        )
        temperature = 0.0
    else:
        prompt = (
            "You are a professional translation engine.\n"
            "Translate Swedish to English.\n\n"
            "Rules:\n"
            "- Translate ALL text, including titles and headings.\n"
            "- Translate compound words literally if needed.\n"
            "- Do NOT preserve Swedish words unless they are personal names.\n"
            "- Output ONLY the English translation.\n"
            "- Keep line breaks, numbers, and punctuation.\n\n"
            f"TEXT:\n{text}"
        )

    payload = {
        "model": MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": temperature}
    }
    resp = requests.post(OLLAMA_URL, json=payload, timeout=timeout_s)
    resp.raise_for_status()
    return resp.json().get("response", "").strip()

def looks_untranslated(inp: str, out: str) -> bool:
    a, b = inp.strip(), out.strip()
    if not b:
        return True
    if a.upper() == b.upper():
        return True
    sw_markers = ["HEMLIGH", "METODEN", "KAPITEL", "ÖVERSÄTT", "SVERIGE", "MINNET"]
    if any(m in b.upper() for m in sw_markers):
        return True
    return False

def translate_text_chunked(sv_text: str) -> str:
    s0 = (sv_text or "").strip()
    if not s0 or len(s0) < 2:
        return sv_text

    chunks = split_for_translation(s0, max_chars=1000)
    out_chunks = []

    for chunk in chunks:
        protected, ph_map = apply_glossary_placeholders(chunk, GLOSSARY)

        out = None
        for attempt in range(3):
            try:
                out = ollama_translate_once(protected, timeout_s=900, temperature=0.1, strong=False)
                out = restore_glossary_placeholders(out, ph_map).strip()
                break
            except (requests.exceptions.ReadTimeout, requests.exceptions.ConnectionError):
                time.sleep(2 * (attempt + 1))

        if out is None:
            try:
                out = ollama_translate_once(protected, timeout_s=1200, strong=True)
                out = restore_glossary_placeholders(out, ph_map).strip()
            except Exception:
                out = chunk + "\n\n[TRANSLATION_FAILED]"

        if looks_untranslated(chunk, out):
            try:
                out2 = ollama_translate_once(protected, timeout_s=1200, strong=True)
                out2 = restore_glossary_placeholders(out2, ph_map).strip()
                if out2 and not looks_untranslated(chunk, out2):
                    out = out2
            except Exception:
                pass

        out_chunks.append(out)

    return "\n\n".join(out_chunks).strip()

def translate_heading_consistently(sv_heading: str) -> str:
    key = sv_heading.strip()
    if key in title_cache:
        return title_cache[key]
    en = translate_text_chunked(key)
    title_cache[key] = en
    save_json(TITLE_CACHE_JSON, title_cache)  # persist immediately
    return en

# ---------- M) Checkpointing (stored on Drive) ----------
def load_state():
    return load_json(STATE_JSON, {"done": 0})

def save_state(done, total=None):
    obj = {"done": done, "updated_at": time.strftime("%Y-%m-%d %H:%M:%S")}
    if total is not None:
        obj["total"] = int(total)
    save_json(STATE_JSON, obj)

def count_total_paragraphs(docx_path: str) -> int:
    doc = Document(docx_path)
    return sum(1 for _ in iter_all_paragraphs(doc))

# ---------- N) Ensure resume artifacts exist on Drive ----------
# If DOCX_EN doesn't exist yet, initialize it from DOCX_SV once.
if not Path(DOCX_EN).exists():
    print("ℹ️ No existing translated DOCX on Drive. Initializing a new one...")
    shutil.copyfile(DOCX_SV, DOCX_EN)
    save_state(0)
    save_json(TITLE_CACHE_JSON, title_cache)
else:
    print("✅ Found existing translated DOCX on Drive:", DOCX_EN)

# Show current state
print("Current STATE:", load_state())
print("DOCX_EN size:", Path(DOCX_EN).stat().st_size, "bytes")

# ---------- O) Translate with progress + ETA + resume ----------
def translate_docx_in_place_with_resume(out_docx: str,
                                       save_every_paragraphs=30,
                                       log_every=200):
    state = load_state()
    done = int(state.get("done", 0))

    total = count_total_paragraphs(out_docx)
    save_state(done, total=total)
    print(f"Total paragraphs: {total} | Resuming from: {done}")

    doc = Document(out_docx)

    times = deque(maxlen=50)
    start_all = time.time()
    changed = 0
    n = 0

    for p in iter_all_paragraphs(doc):
        n += 1
        if n <= done:
            continue

        original = (p.text or "").strip()
        done = n  # always advance state deterministically

        if not original:
            if log_every and (n % log_every == 0):
                pct = 100.0 * n / total
                print(f"{n}/{total} ({pct:.2f}%) [blank]")
            continue

        t0 = time.time()
        if is_heading_like(p):
            translated = translate_heading_consistently(original)
        else:
            translated = translate_text_chunked(original)
        times.append(time.time() - t0)

        replace_paragraph_text_preserve_style(p, translated)
        changed += 1

        if log_every and (n % log_every == 0):
            avg = statistics.mean(times) if times else 0.0
            pct = 100.0 * n / total
            remaining = max(total - n, 0)
            eta_sec = remaining * avg
            elapsed_min = (time.time() - start_all) / 60.0
            print(f"{n}/{total} ({pct:.2f}%) | avg {avg:.2f}s/para | ETA ~ {eta_sec/60:.1f} min | elapsed {elapsed_min:.1f} min")

        if changed >= save_every_paragraphs:
            # Save doc + state directly to Drive (persistent)
            doc.save(out_docx)
            save_state(done, total=total)
            save_json(TITLE_CACHE_JSON, title_cache)
            changed = 0
            pct = 100.0 * done / total
            print(f"💾 Checkpoint saved at #{done} ({pct:.2f}%) to Drive")

    doc.save(out_docx)
    save_state(done, total=total)
    save_json(TITLE_CACHE_JSON, title_cache)
    print(f"✅ Translation complete. Saved to Drive: {out_docx}")
    print(f"✅ Final checkpoint: paragraph #{done}/{total}")

# Run translation (safe to rerun; resumes from Drive state)
translate_docx_in_place_with_resume(
    out_docx=DOCX_EN,
    save_every_paragraphs=30,
    log_every=200
)

# ---------- P) Convert translated DOCX -> PDF ----------
# Convert from Drive DOCX to a local PDF, then copy to Drive
!libreoffice --headless --convert-to pdf --outdir /content "{DOCX_EN}"
if Path(PDF_EN_LOCAL).exists():
    shutil.copy2(PDF_EN_LOCAL, PDF_EN_DRIVE)
    print("✅ PDF saved to Drive:", PDF_EN_DRIVE)
else:
    print("❌ PDF conversion failed; check LibreOffice output/logs.")

# ---------- Q) Download outputs ----------
from google.colab import files
# Download final PDF (and optional DOCX)
if Path(PDF_EN_LOCAL).exists():
    files.download(PDF_EN_LOCAL)
# If you also want the translated DOCX:
files.download(DOCX_EN)


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.0/132.0 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 69.8 MB/s eta 0:00:00
Mounted at /content/drive
✅ Persistent checkpoint dir: /content/drive/MyDrive/superminnet_ckpt
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>